# Expanded ACS missing-feature baseline: ACS 2014 auxiliary only

Complete-feature ACS baseline, PFV reconstruction and uncertainty-aware MF-MIA.
Target: ACS 2023; 16 ACS 2014 shadows. No Adult data are loaded.
Existing target, splits, training, attack settings and asymmetric blind-walk protocol are retained.
PFV samples one nonmissing value from the empirical 2014 marginal distribution.
Seven preselected masks cover one to four missing fields; related predictors remain visible.
The complete-feature attack classifier and auxiliary-selected thresholds remain frozen.
Separate cells add conditional single/average/mean-only/uncertainty-aware comparisons.
Run only after explicit user approval. No model checkpoints or logs are loaded or saved. The historical baseline configuration recorded 62.625% accuracy; a rerun is not guaranteed to reproduce it.


In [1]:
from pathlib import Path
import json
import time
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from src.data import load_acs_income, split_acs_data, subset_target_training
from src.expanded_data import (
    load_expanded_acs, preprocess_expanded_partitions, expanded_feature_inventory,
    reference_split_config, EXPANDED_FEATURES, EXCLUDED, load_expanded_year,
)
from src.training import train_target_model, evaluate_model
from src.shadow_models import (load_adult_data, train_shadow_model, train_shadow_models,
                               summarize_shadow_models, PAPER_SHADOW_ALGORITHMS)
from src.membership_attack import train_membership_classifier, evaluate_attack
from src.paper_attack_features import (collect_shadow_paper_features, collect_target_paper_features,
                                       make_probability_oracle)

torch.set_num_threads(4)
SEED = 42
TRAIN_SIZE = 100_000
INCOME_BASE_YEAR = None
NOMINAL_INCOME_METADATA = dict(enabled=False, label='PINCP > 50000', adjustment='None')
EXPERIMENT_DIR = Path('artifacts/exp_3_2023_2014_100000_nominal_paper_algorithms')
RUN_DIR = EXPERIMENT_DIR / 'reduced_k6_s16_m7'
RUN_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_CONFIG = reference_split_config()
assert SPLIT_CONFIG['seed'] == SEED, 'Use the reference seed consistently.'
TRAIN_CONFIG = dict(hidden_dims=(512, 256, 128), dropout=0.0, epochs=100,
    batch_size=256, learning_rate=1e-3, weight_decay=0.0, seed=SEED,
    device='cpu', early_stopping=False, restore_best=False, monitor_train=True)
WALK_CONFIG = dict(max_steps=20, repeats=2, categorical_change_probability=0.05,
                   batch_size=512, seed=SEED)
SHADOW_RECORDS_PER_CLASS = 2000
N_SHADOW_MODELS = 16
SHADOW_ALGORITHMS = PAPER_SHADOW_ALGORITHMS
# Native commuting units; all other added codes are categorical in the walk.
EXPANDED_NUMERIC_WALK = {'JWMNP': (2.0, 0, 200), 'JWRIP': (1.0, 1, 10)}
print('Training size:', TRAIN_SIZE, '| Input fields:', len(EXPANDED_FEATURES))
print('Reference split configuration:', SPLIT_CONFIG)

import importlib
import src.notebook_output as notebook_output_module
importlib.reload(notebook_output_module)
notebook_output_module.suppress_sklearn_parallel_warning()
from src.notebook_output import CellProgress, quiet_output
assert len(EXPANDED_FEATURES) == 48 and 'VPS' in EXPANDED_FEATURES


Training size: 100000 | Input fields: 48
Reference split configuration: {'validation_size': 10000, 'test_size': 10000, 'mia_nonmember_size': 10000, 'mia_member_size': 10000, 'seed': 42}


## Load 2023 target records and make stratified partitions
Split sizes and seed follow Experiment 2, applied to the new 2023 records.


In [2]:
expanded_source = load_expanded_year(2023, income_base_year=INCOME_BASE_YEAR)
reference_partitions = split_acs_data(expanded_source, **SPLIT_CONFIG)
expanded_base_partitions = reference_partitions
expanded_partitions = subset_target_training(
    expanded_base_partitions, train_size=TRAIN_SIZE, seed=SEED)
for name in ['validation', 'test', 'mia_members', 'mia_nonmembers']:
    np.testing.assert_array_equal(expanded_partitions[name].index, reference_partitions[name].index)
assert set(expanded_partitions['mia_members'].index).issubset(expanded_partitions['train'].index)
for name in ['validation', 'test', 'mia_nonmembers']:
    assert set(expanded_partitions[name].index).isdisjoint(expanded_partitions['train'].index)
print('Target income adjustment:', expanded_source.attrs.get('income_adjustment', NOMINAL_INCOME_METADATA))
print('Eligible records:', len(expanded_source))
print('Unused records:', len(expanded_source) - TRAIN_SIZE
      - sum(len(expanded_partitions[name]) for name in ['validation', 'test', 'mia_nonmembers']))
display(pd.DataFrame({name: {'records': len(part), 'positive_rate': part['income_gt_50k'].mean()}
    for name, part in expanded_partitions.items()}).T)
feature_inventory = expanded_feature_inventory(expanded_partitions['train'])
display(feature_inventory)


Target income adjustment: {'enabled': False, 'label': 'PINCP > 50000', 'adjustment': 'None'}
Eligible records: 203278
Unused records: 73278


,records,positive_rate
validation,10000.0,0.49840
test,10000.0,0.49840
mia_nonmembers,10000.0,0.49840
train,100000.0,0.49845
mia_members,10000.0,0.49840


,group,type,unique_nonmissing,missing_fraction
feature,,,,
AGEP,original_ACSIncome,numeric,75,0.00000
COW,original_ACSIncome,categorical,8,0.00000
SCHL,original_ACSIncome,categorical,24,0.00000
MAR,original_ACSIncome,categorical,5,0.00000
OCCP,original_ACSIncome,categorical,527,0.00000
POBP,original_ACSIncome,categorical,220,0.00000
WKHP,original_ACSIncome,numeric,97,0.00000
SEX,original_ACSIncome,categorical,2,0.00000
RAC1P,original_ACSIncome,categorical,9,0.00000


## Preprocess using training records only
The output is sparse; the MLP densifies only the current batch. Cardinality
counts above refer to the original fields, not one-hot dimensions.


In [3]:
expanded_preprocessor, expanded_arrays = preprocess_expanded_partitions(expanded_partitions)
print('Encoded input dimensions:', expanded_arrays['train']['X'].shape[1])
print('Training CSR storage (MiB):', round(sum(
    value.nbytes for value in [expanded_arrays['train']['X'].data,
    expanded_arrays['train']['X'].indices, expanded_arrays['train']['X'].indptr]) / 2**20, 2))
pd.DataFrame({name: {'rows': values['X'].shape[0], 'encoded_features': values['X'].shape[1]}
    for name, values in expanded_arrays.items()}).T


Encoded input dimensions: 2689
Training CSR storage (MiB): 38.53


,rows,encoded_features
validation,10000,2689
test,10000,2689
mia_nonmembers,10000,2689
train,100000,2689
mia_members,10000,2689


## Train the target
Fixed 100 epochs, final weights retained. Validation is monitored but does not
select an epoch in this deliberately overfit experimental condition.


In [4]:
target_started = time.monotonic()
with quiet_output():
    expanded_model, expanded_history, expanded_metadata = train_target_model(
        expanded_arrays['train']['X'], expanded_arrays['train']['y'],
        expanded_arrays['validation']['X'], expanded_arrays['validation']['y'], **TRAIN_CONFIG)
target_training_seconds = time.monotonic() - target_started
print('Target training seconds:', round(target_training_seconds, 1))
display(pd.DataFrame(expanded_history).iloc[::10])


Target training seconds: 963.4


,epoch,train_loss,validation_accuracy,validation_roc_auc,validation_log_loss,train_eval_accuracy,train_eval_roc_auc,train_eval_log_loss
0,1,0.416451,0.8134,0.900563,0.403865,0.82923,0.913752,0.378947
10,11,0.060908,0.7923,0.867702,1.158196,0.98532,0.999131,0.039848
20,21,0.016433,0.7787,0.858233,1.512833,0.99233,0.999692,0.021763
30,31,0.011112,0.7940,0.868656,1.540580,0.99794,0.999976,0.006108
40,41,0.007754,0.7937,0.868837,1.735596,0.99766,0.999973,0.006455
50,51,0.007198,0.7898,0.869194,1.748980,0.99778,0.999985,0.005592
60,61,0.004103,0.7855,0.862731,1.791714,0.99801,0.999974,0.006090
70,71,0.005068,0.7877,0.863999,1.846895,0.99766,0.999985,0.006761
80,81,0.003399,0.7911,0.867223,1.839046,0.99867,0.999990,0.003930
90,91,0.003514,0.7906,0.865856,1.933827,0.99936,0.999998,0.001810


## Income-prediction evaluation
These are target income metrics, not membership metrics. The test set was not
used for model selection. Do not choose target settings using these results.


In [5]:
expanded_metrics = {name: evaluate_model(expanded_model, expanded_arrays[name]['X'],
    expanded_arrays[name]['y']) for name in ['train', 'validation', 'test']}
display(pd.DataFrame(expanded_metrics).T)
print('Train-test accuracy gap (percentage points):',
      100 * (expanded_metrics['train']['accuracy'] - expanded_metrics['test']['accuracy']))


,accuracy,roc_auc,log_loss
train,0.99849,0.999985,0.004646
validation,0.78650,0.861110,1.959403
test,0.79520,0.867629,1.864808


Train-test accuracy gap (percentage points): 20.328999999999997


## 16 expanded ACS 2014 shadows using the paper's three named algorithms
Keep existing RF/XGBoost settings and 40/20/40 splits.
Gradient boosting uses 300 trees, learning rate 0.05, maximum depth 3, seed per shadow.
These are project settings; the paper does not provide its exact configurations.
Each preprocessor fits its own training records.
These shadows do not assume knowledge of the target architecture.
Cycle the three algorithms using unique seeds and splits. Models may reuse auxiliary
records across shadows; grouped attack splits keep repeated records together.


In [6]:
acs_auxiliary = load_expanded_year(2014, income_base_year=INCOME_BASE_YEAR)
print('Auxiliary income adjustment:', acs_auxiliary.attrs.get('income_adjustment', NOMINAL_INCOME_METADATA))
assert set(acs_auxiliary.index).isdisjoint(expanded_source.index)
shadow_results = {}
shadow_progress = CellProgress()
for index in range(N_SHADOW_MODELS):
    algorithm = SHADOW_ALGORITHMS[index % len(SHADOW_ALGORITHMS)]
    name = f'{algorithm}_{index + 1:02d}'
    shadow_progress.update(f'Training shadow {index + 1}/{N_SHADOW_MODELS}: {name}')
    with quiet_output():
        result = train_shadow_model(acs_auxiliary, algorithm, seed=SEED + 10 * index, schema='expanded_acs')
    result['attack_data']['shadow_model'] = name
    shadow_results[name] = result
shadow_progress.update(f'Completed {len(shadow_results)} shadows')
# Verify native-input queries reproduce probabilities computed during training.
for name, result in shadow_results.items():
    audit_rows = result['attack_data'].iloc[:32]
    native = acs_auxiliary.loc[audit_rows['record_id'], EXPANDED_FEATURES]
    query = make_probability_oracle(result['model'], result['preprocessor'], schema='acs')
    np.testing.assert_allclose(query(native),
        audit_rows[['p_income_le_50k', 'p_income_gt_50k']].to_numpy(),
        rtol=1e-5, atol=1e-6, err_msg=f'Query mismatch for {name}')
print('Probability oracle consistency verified for all 16 shadows.')
display(summarize_shadow_models(shadow_results))


Auxiliary income adjustment: {'enabled': False, 'label': 'PINCP > 50000', 'adjustment': 'None'}


Probability oracle consistency verified for all 16 shadows.


records  accuracy   roc_auc  log_loss
shadow_model         partition                                        
random_forest_01     train         73576  0.895034  0.963991  0.305143
                     validation    36789  0.822447  0.898100  0.396220
                     nonmembers    73576  0.823815  0.899010  0.395105
gradient_boosting_02 train         73576  0.828857  0.905512  0.374626
                     validation    36789  0.828046  0.902338  0.379613
                     nonmembers    73576  0.825092  0.902493  0.379359
xgboost_03           train         73576  0.841076  0.916833  0.352353
                     validation    36789  0.834489  0.911705  0.360892
                     nonmembers    73576  0.833669  0.909643  0.364265
random_forest_04     train         73576  0.895876  0.964020  0.306229
                     validation    36789  0.826307  0.900072  0.393914
                     nonmembers    73576  0.824019  0.898980  0.395291
gradient_boosting_05 train         73576  0.828776  0.906059  0.373634
                     validation    36789  0.826307  0.903606  0.377615
                     nonmembers    73576  0.826696  0.902084  0.379925
xgboost_06           train         73576  0.843645  0.918923  0.348288
                     validation    36789  0.833592  0.909114  0.364757
                     nonmembers    73576  0.832255  0.908488  0.365835
random_forest_07     train         73576  0.893851  0.963988  0.304787
                     validation    36789  0.824350  0.900185  0.392477
                     nonmembers    73576  0.821137  0.897831  0.396558
gradient_boosting_08 train         73576  0.830842  0.907635  0.370956
                     validation    36789  0.823942  0.901067  0.381202
                     nonmembers    73576  0.824603  0.901087  0.381115
xgboost_09           train         73576  0.842326  0.918027  0.349982
                     validation    36789  0.833374  0.907983  0.366911
                     nonmembers    73576  0.834267  0.910003  0.363418
random_forest_10     train         73576  0.895496  0.963504  0.305802
                     validation    36789  0.824621  0.896990  0.397782
                     nonmembers    73576  0.826248  0.901203  0.392587
gradient_boosting_11 train         73576  0.828205  0.905770  0.374374
                     validation    36789  0.824051  0.901651  0.380322
                     nonmembers    73576  0.826248  0.902440  0.379078
xgboost_12           train         73576  0.841538  0.918526  0.349140
                     validation    36789  0.834869  0.909549  0.364243
                     nonmembers    73576  0.832432  0.908971  0.364974
random_forest_13     train         73576  0.893077  0.963590  0.305655
                     validation    36789  0.823453  0.900789  0.392610
                     nonmembers    73576  0.823665  0.898826  0.394883
gradient_boosting_14 train         73576  0.828545  0.905773  0.374242
                     validation    36789  0.825437  0.901572  0.380655
                     nonmembers    73576  0.825894  0.902501  0.379467
xgboost_15           train         73576  0.842394  0.918390  0.349351
                     validation    36789  0.833836  0.908037  0.366705
                     nonmembers    73576  0.832432  0.908918  0.364909
random_forest_16     train         73576  0.894626  0.964139  0.305157
                     validation    36789  0.825682  0.899839  0.394492
                     nonmembers    73576  0.824046  0.899143  0.394792

## Extract shadow features and fit the random forest attack
Features: class-scaled logit, blind walk, entropy and cross-entropy. Use 2,000
members and 2,000 non-members per shadow. Attack record-ID grouping prevents
repeated 2014 records from crossing attack partitions. Select both thresholds
on auxiliary validation only; no target membership labels train this classifier.


In [7]:
with quiet_output():
    shadow_attack_data, shadow_query_diagnostics = collect_shadow_paper_features(
        shadow_results, acs_auxiliary, per_class=SHADOW_RECORDS_PER_CLASS,
        config=WALK_CONFIG, seed=SEED)
display(pd.DataFrame(shadow_query_diagnostics).T)
attack_result = train_membership_classifier(
    shadow_attack_data, seed=SEED, algorithm='random_forest', feature_set='paper')
display(pd.DataFrame(attack_result['metrics']).T)


,record_queries,max_record_queries,walk_censored_fraction,blind_walk_enabled,config
random_forest_01,126429,164000,0.643375,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
gradient_boosting_02,124549,164000,0.613375,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
xgboost_03,124439,164000,0.616375,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
random_forest_04,126876,164000,0.647,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
gradient_boosting_05,123461,164000,0.60925,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
xgboost_06,124575,164000,0.609,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
random_forest_07,127145,164000,0.645875,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
gradient_boosting_08,123744,164000,0.609125,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
xgboost_09,124469,164000,0.615125,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."
random_forest_10,127005,164000,0.647125,True,"{'max_steps': 20, 'repeats': 2, 'categorical_c..."


,records,members,accuracy,balanced_accuracy,roc_auc,tpr_at_1pct_fpr_roc,threshold,tn,fp,fn,tp,shadow_selected_threshold,tpr_at_shadow_threshold,fpr_at_shadow_threshold
train,38419.0,19317.0,0.601942,0.601382,0.646115,0.032769,0.507089,9577.0,9525.0,5768.0,13549.0,0.596605,0.015945,0.004031
validation,12849.0,6376.0,0.519496,0.520273,0.521083,0.012704,0.507089,2701.0,3772.0,2402.0,3974.0,0.596605,0.012704,0.009887
test,12732.0,6307.0,0.511074,0.512006,0.520823,0.012526,0.507089,2644.0,3781.0,2444.0,3863.0,0.596605,0.011099,0.009183


## Final MIA evaluation on the expanded target
Query the expanded native input schema, preserving categorical codes and
handling numeric commute fields in native units. The walk still generates
field-valid but potentially implausible combinations, and its dimensionality
matches the expanded auxiliary schema. Thresholds are frozen from auxiliary validation.


In [8]:
with quiet_output():
    target_attack_data, target_query_diagnostics = collect_target_paper_features(
        expanded_model, expanded_preprocessor, expanded_partitions,
        config=WALK_CONFIG, seed=SEED, numeric_protocol=EXPANDED_NUMERIC_WALK)
display(pd.Series(target_query_diagnostics, name='expanded target query diagnostics'))
target_mia_metrics, target_membership_scores = evaluate_attack(
    attack_result['classifier'], target_attack_data,
    threshold=attack_result['accuracy_threshold'],
    low_fpr_threshold=attack_result['low_fpr_threshold'])
target_mia_metrics['tpr_at_accuracy_threshold'] = target_mia_metrics['tp'] / target_mia_metrics['members']
target_mia_metrics['fpr_at_accuracy_threshold'] = target_mia_metrics['fp'] / (
    target_mia_metrics['records'] - target_mia_metrics['members'])
display(pd.DataFrame({'2014 auxiliary attack test': attack_result['metrics']['test'],
                      'Expanded ACS target MIA': target_mia_metrics}).T)


record_queries                                                       526991
max_record_queries                                                   820000
walk_censored_fraction                                             0.438975
blind_walk_enabled                                                     True
config                    {'max_steps': 20, 'repeats': 2, 'categorical_c...
Name: expanded target query diagnostics, dtype: object

,records,members,accuracy,balanced_accuracy,roc_auc,tpr_at_1pct_fpr_roc,threshold,tn,fp,fn,tp,shadow_selected_threshold,tpr_at_shadow_threshold,fpr_at_shadow_threshold,tpr_at_accuracy_threshold,fpr_at_accuracy_threshold
2014 auxiliary attack test,12732.0,6307.0,0.511074,0.512006,0.520823,0.012526,0.507089,2644.0,3781.0,2444.0,3863.0,0.596605,0.011099,0.009183,NaN,NaN
Expanded ACS target MIA,20000.0,10000.0,0.626250,0.626250,0.608622,0.001700,0.507089,2724.0,7276.0,199.0,9801.0,0.596605,0.001400,0.004500,0.9801,0.7276


## Small reproducibility artifacts
Save summaries, the feature inventory and row assignments. Models remain in memory;
no checkpoints or log files are loaded or saved.


In [9]:
result_dir = RUN_DIR
result_dir.mkdir(parents=True, exist_ok=True)
feature_inventory.to_csv(result_dir / 'feature_inventory.csv')
for name, part in expanded_partitions.items():
    pd.Series(part.index, name='record_id').to_csv(result_dir / f'{name}_ids.csv', index=False)
summarize_shadow_models(shadow_results).to_csv(result_dir / 'shadow_metrics.csv')
summary = dict(income_base_year=INCOME_BASE_YEAR,
    target_income_adjustment=expanded_source.attrs.get('income_adjustment', NOMINAL_INCOME_METADATA),
    auxiliary_income_adjustment=acs_auxiliary.attrs.get('income_adjustment', NOMINAL_INCOME_METADATA),
    perturbation_protocol_audit={'shadow_numeric_fields': {'AGEP': [1.0, 17, 99], 'WKHP': [2.0, 1, 99]}, 'target_extra_numeric_fields': {'JWMNP': [2.0, 0, 200], 'JWRIP': [1.0, 1, 10]}, 'shadow_commuting_fields': 'Categorical replacements, unchanged from original run', 'target_commuting_fields': 'Numeric perturbations, unchanged from original run', 'correction': 'Prior assistant statements that the shadow commuting perturbation protocol changed were incorrect. Original and current perturbation call sources are identical.'},
    walk_numeric_protocol_scope='Extra target numeric fields only',
    walk_numeric_protocol=EXPANDED_NUMERIC_WALK, execution_status='acs_baseline_completed',
    attack_metrics_status='oracle_consistency_verified', target_year=2023, auxiliary_year=2014, n_shadow_models=N_SHADOW_MODELS,
    shadow_algorithms=list(SHADOW_ALGORITHMS),
    shadow_seeds={name: result['seed'] for name, result in shadow_results.items()},
    original_feature_count=len(EXPANDED_FEATURES),
    encoded_feature_count=expanded_arrays['train']['X'].shape[1],
    train_size=TRAIN_SIZE, split_config=SPLIT_CONFIG, training=expanded_metadata,
    target_training_seconds=target_training_seconds, target_metrics=expanded_metrics,
    shadow_attack_metrics=attack_result['metrics'], target_mia_metrics=target_mia_metrics,
    target_query_diagnostics=target_query_diagnostics)
(result_dir / 'summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print('Summary and exact row IDs written to', result_dir)


Summary and exact row IDs written to artifacts\exp_3_2023_2014_100000_nominal_paper_algorithms\reduced_k6_s16_m7


## Missing-feature PFV baseline
Use seven preselected masks: two singles, two pairs, two triples and one quadruple.
These retain the reduced K=6 study; the complete-feature baseline uses the historical settings.
All 48 target inputs remain required; the target is not retrained after masking.
Each hidden field is replaced before querying, including before blind walks.
The reconstruction pool contains ACS 2014 predictors only, with no income-label
conditioning. Naturally missing auxiliary values are excluded from the sampling pool.
Use stable-code fields only; do not extend this to occupation/industry without
an explicit cross-year category mapping. Related marriage and employment fields
remain observed. Income labels remain known for the original attack's loss features.

The same complete-feature shadow attack is used in every condition, isolating the
effect of target completion. There is no mask-specific attack retraining. Auxiliary
attack train/validation/test grouping and thresholds therefore remain unchanged.
A single seed is a baseline realization, not a measure of reconstruction variability.


In [10]:
from itertools import combinations
from src.missing_feature_pfv import complete_pfv

MASK_FIELDS = ['MAR', 'WKHP', 'COW', 'SEX']
MISSING_MASKS = {
    'MAR': ['MAR'], 'WKHP': ['WKHP'],
    'MAR+COW': ['MAR', 'COW'], 'COW+SEX': ['COW', 'SEX'],
    'MAR+WKHP+SEX': ['MAR', 'WKHP', 'SEX'],
    'WKHP+COW+SEX': ['WKHP', 'COW', 'SEX'],
    'MAR+WKHP+COW+SEX': ['MAR', 'WKHP', 'COW', 'SEX'],
}
MISSING_DIR = RUN_DIR / 'missing_feature_pfv'
MISSING_DIR.mkdir(parents=True, exist_ok=True)
missing_rows = [{'condition': 'complete', 'n_missing': 0, **target_mia_metrics}]
missing_audits = {}
for mask_name, hidden in MISSING_MASKS.items():
    completed_partitions = {}
    # One shared completion pass: membership never selects a seed or distribution.
    candidates = pd.concat([expanded_partitions[name] for name in
        ['mia_members', 'mia_nonmembers']]).sort_index()
    completed_all = complete_pfv(candidates[EXPANDED_FEATURES],
        acs_auxiliary[EXPANDED_FEATURES], hidden, seed=SEED)
    completed_all['income_gt_50k'] = candidates['income_gt_50k']
    for name in ['mia_members', 'mia_nonmembers']:
        original = expanded_partitions[name]
        completed_partitions[name] = completed_all.loc[original.index].copy()
    with quiet_output():
        features, audit = collect_target_paper_features(
            expanded_model, expanded_preprocessor, completed_partitions,
            config=WALK_CONFIG, seed=SEED, numeric_protocol=EXPANDED_NUMERIC_WALK)
    metrics, scores = evaluate_attack(attack_result['classifier'], features,
        threshold=attack_result['accuracy_threshold'],
        low_fpr_threshold=attack_result['low_fpr_threshold'])
    metrics['tpr_at_accuracy_threshold'] = metrics['tp'] / metrics['members']
    metrics['fpr_at_accuracy_threshold'] = metrics['fp'] / (metrics['records'] - metrics['members'])
    missing_rows.append({'condition': mask_name, 'n_missing': len(hidden), **metrics})
    missing_audits[mask_name] = audit
    features.assign(membership_score=scores).to_csv(MISSING_DIR / f'{mask_name}_scores.csv', index=False)
missing_comparison = pd.DataFrame(missing_rows).set_index('condition')
missing_comparison['auc_change_from_complete'] = missing_comparison['roc_auc'] - target_mia_metrics['roc_auc']
display(missing_comparison)
missing_comparison.to_csv(MISSING_DIR / 'comparison.csv')
(MISSING_DIR / 'protocol.json').write_text(json.dumps(dict(
    method='PFV_single_draw', auxiliary_year=2014, target_year=2023,
    masks=MISSING_MASKS, seed=SEED,
    naturally_missing_auxiliary_values='excluded',
    classifier='frozen complete-feature ACS 2014 attack',
    thresholds='frozen auxiliary validation', diagnostics=missing_audits,
    baseline_summary='../summary.json'), indent=2), encoding='utf-8')


,n_missing,records,members,accuracy,balanced_accuracy,roc_auc,tpr_at_1pct_fpr_roc,threshold,tn,fp,fn,tp,shadow_selected_threshold,tpr_at_shadow_threshold,fpr_at_shadow_threshold,tpr_at_accuracy_threshold,fpr_at_accuracy_threshold,auc_change_from_complete
condition,,,,,,,,,,,,,,,,,,
complete,0,20000,10000,0.62625,0.62625,0.608622,0.0017,0.507089,2724,7276,199,9801,0.596605,0.0014,0.0045,0.9801,0.7276,0.000000
MAR,1,20000,10000,0.61870,0.61870,0.603223,0.0029,0.507089,2798,7202,424,9576,0.596605,0.0021,0.0059,0.9576,0.7202,-0.005400
WKHP,1,20000,10000,0.59590,0.59590,0.586256,0.0044,0.507089,3170,6830,1252,8748,0.596605,0.0037,0.0045,0.8748,0.6830,-0.022367
MAR+COW,2,20000,10000,0.60925,0.60925,0.594035,0.0033,0.507089,2891,7109,706,9294,0.596605,0.0022,0.0062,0.9294,0.7109,-0.014588
COW+SEX,2,20000,10000,0.60445,0.60445,0.593877,0.0032,0.507089,2812,7188,723,9277,0.596605,0.0024,0.0050,0.9277,0.7188,-0.014745
MAR+WKHP+SEX,3,20000,10000,0.58815,0.58815,0.579768,0.0045,0.507089,3352,6648,1589,8411,0.596605,0.0036,0.0052,0.8411,0.6648,-0.028855
WKHP+COW+SEX,3,20000,10000,0.58070,0.58070,0.574166,0.0063,0.507089,3248,6752,1634,8366,0.596605,0.0051,0.0047,0.8366,0.6752,-0.034456
MAR+WKHP+COW+SEX,4,20000,10000,0.57805,0.57805,0.571844,0.0044,0.507089,3418,6582,1857,8143,0.596605,0.0038,0.0058,0.8143,0.6582,-0.036779


3069

## Uncertainty-aware MF-MIA: study protocol
Research question: does reconstruction-induced score standard deviation improve
membership inference beyond the mean? The following cells are additional experiments;
they are not executed when this notebook is generated.

Use the seven preselected masks, target, 16 ACS 2014 shadows, four base
attack features and existing blind-walk protocol. The base attack is frozen.
Compare conditional single imputation, average score, learned mean-only RF and
matched mean+standard-deviation RF on identical completion draws. PFV above is a
separate reconstruction baseline, not the single-imputation control for this section.

Sequential forest-leaf donor sampling conditions each hidden field on visible
fields and earlier sampled hidden fields (MAR, WKHP, COW, SEX order). It preserves
observed auxiliary values but is an approximate predictive distribution, not a
calibrated posterior. Reconstruction training excludes all base-attack, meta-fit,
calibration and auxiliary-test candidate IDs. No income label conditions imputation.
Auxiliary validation IDs are divided into meta-fit and calibration groups. Auxiliary
test IDs remain untouched until evaluation. The base attack's earlier validation
thresholds are not reused for this comparison. No target labels select methods,
masks, K or thresholds. Income class labels remain known to the base loss feature.

K=6 and one reconstruction seed are predeclared exploratory settings, not a
confirmatory multi-seed study. All seven masks are reported without choosing a winner
on target results. Raw draws remain in memory during this run.
The original commuting walk asymmetry is retained and limits interpretation.
Use two single-field, two double-field, two triple-field and one four-field mask;
each of MAR, WKHP, COW and SEX is hidden in four masks. Six draws give noisier
variability estimates than the original K=20 setting. The base attack is refitted
using all 16 shadows, with grouped auxiliary splits recomputed from
that pool. Reconstruction/meta/calibration roles follow those new splits.
Results are saved under reduced_k6_s16_m7, separate from the eight-shadow run and full experiment.
Common walk seeds reduce additional Monte Carlo noise; score spread can still
include completion-dependent effects of the finite blind-walk procedure.

This section can be expensive: each candidate is evaluated K times per mask,
with up to 41 record queries per completion under the original walk settings.
DP training/evaluation and repeated-seed confirmation are not implemented here.

Execution uses memory only for reconstruction models, audit draws, completion
scores and aggregators. No checkpoints are loaded or saved. Run these cells in
order in one kernel session; restarting requires computing them again.
Only final metrics, row assignments and the protocol are written to disk.


In [11]:
# 1. Fixed configuration and disjoint auxiliary roles
import importlib
from sklearn.model_selection import train_test_split
from src.notebook_output import CellProgress, quiet_output
import src.uncertainty_mf_mia as uncertainty_module
# Pick up runtime-support edits without retraining/reloading the target or shadows.
importlib.reload(uncertainty_module)
from src.uncertainty_mf_mia import (
    ConditionalDonorChain, auxiliary_roles, completion_scores, summarize_scores,
    fit_aggregation, evaluate_aggregation,
)

UNCERTAINTY_K = 6
assert N_SHADOW_MODELS == len(shadow_results) == 16, 'Load the 16-shadow pool first.'
assert set(shadow_attack_data['shadow_model']) == set(shadow_results), 'Rebuild base attack data using all 16 shadows.'
assert set().union(*(set(part['shadow_model']) for part in attack_result['splits'].values())) == set(shadow_results), 'Refit the base attack on the 16-shadow pool.'
assert len(MISSING_MASKS) == 7, 'Use the seven preselected masks for the reduced study.'
RECONSTRUCTION_SEED = SEED + 1000
UNCERTAINTY_DIR = RUN_DIR / 'uncertainty_mf_mia'
UNCERTAINTY_DIR.mkdir(parents=True, exist_ok=True)
uncertainty_roles = auxiliary_roles(acs_auxiliary, attack_result, seed=SEED)
recon_fit_ids, recon_check_ids = train_test_split(
    sorted(uncertainty_roles['reconstruction']), test_size=0.2, random_state=SEED)
recon_training = acs_auxiliary.loc[recon_fit_ids, EXPANDED_FEATURES]
recon_check = acs_auxiliary.loc[recon_check_ids, EXPANDED_FEATURES]
display(pd.Series({name: len(ids) for name, ids in uncertainty_roles.items()}, name='unique_2014_records'))
uncertainty_reconstructors = {}


reconstruction    131002
base               31763
meta                5294
calibration         5294
test               10588
Name: unique_2014_records, dtype: int64

### Fit and audit conditional reconstruction using 2014 only
Held-out reconstruction checks use only reserved 2014 records, not attack candidates.
Numeric coverage is descriptive empirical coverage of the sampled 90% interval.
Categorical sample-match fraction measures draw accuracy; it is not calibrated
probability accuracy. Report these checks even when attack performance is poor.
Related fields remain visible, so reconstruction may be easy through proxies.


In [12]:
# 2. Fit one conditional reconstruction chain per mask; no target queries
from src.notebook_output import CellProgress, quiet_output
reconstruction_audit_rows = []
reconstruction_progress = CellProgress()
for mask_index, (mask_name, hidden) in enumerate(MISSING_MASKS.items(), start=1):
    mask_started = time.monotonic()
    reconstruction_progress.update(f'[{mask_index}/{len(MISSING_MASKS)}] {mask_name}: fitting reconstruction')
    with quiet_output():
        reconstructor = ConditionalDonorChain(hidden, seed=RECONSTRUCTION_SEED).fit(recon_training)
    uncertainty_reconstructors[mask_name] = reconstructor
    reconstruction_progress.update(f'[{mask_index}/{len(MISSING_MASKS)}] {mask_name}: auditing {UNCERTAINTY_K} completions')
    with quiet_output():
        samples = list(reconstructor.iter_samples(recon_check,
            seeds=[RECONSTRUCTION_SEED + draw for draw in range(UNCERTAINTY_K)]))
    for field in hidden:
        observed = recon_check[field].notna().to_numpy()
        if not observed.any():
            continue
        draws = np.column_stack([sample[field].to_numpy()[observed] for sample in samples])
        truth = recon_check[field].to_numpy()[observed]
        row = dict(mask=mask_name, field=field, records=int(observed.sum()))
        if field == 'WKHP':
            lower, upper = np.quantile(draws, [0.05, 0.95], axis=1)
            row.update(mae_of_sample_mean=float(np.abs(draws.mean(axis=1) - truth).mean()),
                sampled_90pct_coverage=float(((truth >= lower) & (truth <= upper)).mean()),
                mean_interval_width=float((upper - lower).mean()))
        else:
            row['sample_match_fraction'] = float((draws == truth[:, None]).mean())
        reconstruction_audit_rows.append(row)
    reconstruction_progress.update(f'[{mask_index}/{len(MISSING_MASKS)}] {mask_name}: complete ({time.monotonic() - mask_started:.1f}s)')
reconstruction_audit = pd.DataFrame(reconstruction_audit_rows)
display(reconstruction_audit.round(4))
reconstruction_audit.to_csv(UNCERTAINTY_DIR / 'reconstruction_audit.csv', index=False)


,mask,field,records,sample_match_fraction,mae_of_sample_mean,sampled_90pct_coverage,mean_interval_width
0,MAR,MAR,26201,0.6076,NaN,NaN,NaN
1,WKHP,WKHP,26201,NaN,8.6220,0.7338,22.5111
2,MAR+COW,MAR,26201,0.6278,NaN,NaN,NaN
3,MAR+COW,COW,26201,0.4971,NaN,NaN,NaN
4,COW+SEX,COW,26201,0.5009,NaN,NaN,NaN
5,COW+SEX,SEX,26201,0.5390,NaN,NaN,NaN
6,MAR+WKHP+SEX,MAR,26201,0.6354,NaN,NaN,NaN
7,MAR+WKHP+SEX,WKHP,26201,NaN,8.7223,0.7316,22.9166
8,MAR+WKHP+SEX,SEX,26201,0.5384,NaN,NaN,NaN
9,WKHP+COW+SEX,WKHP,26201,NaN,8.7994,0.7281,23.1203


### Generate held-out auxiliary completion scores
The original 2,000 member/nonmember samples per shadow are reused. Only meta-fit,
calibration and auxiliary-test records are queried here; base-attack training
records are excluded. Repeated IDs retain the same role across all shadows.
Completion never receives membership labels as a conditioning variable.


In [13]:
# 3. K scores per held-out shadow observation, separately for every mask
from src.notebook_output import CellProgress, quiet_output
uncertainty_auxiliary_tables = {}
uncertainty_query_audits = {}
auxiliary_progress = CellProgress()
heldout_ids = set().union(*(uncertainty_roles[role] for role in ['meta', 'calibration', 'test']))
for mask_name, reconstructor in uncertainty_reconstructors.items():
    mask_tables = []
    for shadow_name, shadow in shadow_results.items():
        selected = shadow_attack_data.loc[
            (shadow_attack_data['shadow_model'] == shadow_name)
            & shadow_attack_data['record_id'].isin(heldout_ids),
            ['record_id', 'membership']].sort_values('record_id').reset_index(drop=True)
        if selected.empty:
            continue
        assert selected['record_id'].is_unique
        candidates = acs_auxiliary.loc[selected['record_id']]
        oracle = make_probability_oracle(shadow['model'], shadow['preprocessor'], schema='acs')
        auxiliary_progress.update(f'Auxiliary scoring: {mask_name} / {shadow_name}, {len(selected):,} records, K={UNCERTAINTY_K}')
        with quiet_output():
            draw_scores, audit = completion_scores(oracle, candidates, reconstructor,
                attack_result['classifier'], k=UNCERTAINTY_K, seed=RECONSTRUCTION_SEED,
                walk_config=WALK_CONFIG)  # preserve existing shadow walk protocol
        table = summarize_scores(draw_scores)
        table['record_id'] = selected['record_id'].to_numpy()
        table['membership'] = selected['membership'].to_numpy()
        table['shadow_model'] = shadow_name
        table['role'] = [next(role for role in ['meta', 'calibration', 'test']
            if record_id in uncertainty_roles[role]) for record_id in table['record_id']]
        mask_tables.append(table)
        uncertainty_query_audits[f'{mask_name}/{shadow_name}'] = audit
    uncertainty_auxiliary_tables[mask_name] = pd.concat(mask_tables, ignore_index=True)
auxiliary_progress.update(f'Auxiliary scoring complete: {len(uncertainty_auxiliary_tables)} masks')


In [14]:
# 4. Fit aggregators, freeze thresholds on calibration, evaluate auxiliary test
from src.notebook_output import quiet_output
uncertainty_aggregators = {}
uncertainty_auxiliary_result_rows = []
for mask_name, table in uncertainty_auxiliary_tables.items():
    meta = table.loc[table['role'] == 'meta']
    calibration = table.loc[table['role'] == 'calibration']
    auxiliary_test = table.loc[table['role'] == 'test']
    with quiet_output():
        learned, thresholds = fit_aggregation(meta, calibration, seed=SEED)
    uncertainty_aggregators[mask_name] = (learned, thresholds)
    with quiet_output():
        metrics, predictions = evaluate_aggregation(auxiliary_test, learned, thresholds)
    uncertainty_auxiliary_result_rows.extend(metrics.assign(mask=mask_name,
        n_missing=len(MISSING_MASKS[mask_name]), dataset='2014_auxiliary_test').to_dict('records'))
print('Auxiliary test: macro averages across masks of the same size')
display(pd.DataFrame(uncertainty_auxiliary_result_rows).groupby(['n_missing', 'method'])[
    ['roc_auc', 'balanced_accuracy', 'tpr_at_1pct_fpr_roc']].mean().round(4))


Auxiliary test: macro averages across masks of the same size


roc_auc  balanced_accuracy  tpr_at_1pct_fpr_roc
n_missing method                                                       
1         average        0.5207             0.5141               0.0109
          learned_mean   0.4992             0.4996               0.0096
          single         0.5179             0.5162               0.0101
          uncertainty    0.5056             0.5046               0.0083
2         average        0.5130             0.5071               0.0112
          learned_mean   0.5051             0.5007               0.0101
          single         0.5071             0.5073               0.0105
          uncertainty    0.5079             0.5057               0.0094
3         average        0.5204             0.5129               0.0113
          learned_mean   0.5025             0.5010               0.0076
          single         0.5151             0.5086               0.0128
          uncertainty    0.4970             0.4977               0.0077
4         average        0.5132             0.5097               0.0114
          learned_mean   0.5087             0.5058               0.0100
          single         0.5129             0.5080               0.0122
          uncertainty    0.5045             0.5003               0.0128

### Final target evaluation: frozen methods and thresholds
Use the same target member/nonmember IDs as the complete-feature baseline.
Membership labels are attached only after scores are generated. The same K draws
feed single, averaging and both learned aggregators. Do not tune on these results.
The complete-feature and PFV results above are context; the controlled test for
the standard-deviation signal compares uncertainty against learned_mean here.


In [15]:
# 5. Query the existing ACS 2023 target; no target retraining
from src.notebook_output import CellProgress, quiet_output
uncertainty_result_rows = list(uncertainty_auxiliary_result_rows)
target_candidates = pd.concat([expanded_partitions[name]
    for name in ['mia_members', 'mia_nonmembers']]).sort_index()
target_member_ids = set(expanded_partitions['mia_members'].index)
target_oracle = make_probability_oracle(expanded_model, expanded_preprocessor, schema='acs')
target_progress = CellProgress()
for mask_name, reconstructor in uncertainty_reconstructors.items():
    target_progress.update(f'Target scoring: {mask_name}, K={UNCERTAINTY_K}')
    with quiet_output():
        draw_scores, audit = completion_scores(target_oracle, target_candidates,
            reconstructor, attack_result['classifier'], k=UNCERTAINTY_K,
            seed=RECONSTRUCTION_SEED, walk_config=WALK_CONFIG,
            numeric_protocol=EXPANDED_NUMERIC_WALK)
    table = summarize_scores(draw_scores)
    table['record_id'] = target_candidates.index.to_numpy()
    table['membership'] = table['record_id'].isin(target_member_ids).astype(int)
    learned, thresholds = uncertainty_aggregators[mask_name]
    with quiet_output():
        metrics, predictions = evaluate_aggregation(table, learned, thresholds)
    uncertainty_result_rows.extend(metrics.assign(mask=mask_name,
        n_missing=len(MISSING_MASKS[mask_name]), dataset='2023_target').to_dict('records'))
    uncertainty_query_audits[f'{mask_name}/target'] = audit
uncertainty_results = pd.DataFrame(uncertainty_result_rows)
uncertainty_results.to_csv(UNCERTAINTY_DIR / 'comparison.csv', index=False)
target_progress.update(f'Target scoring complete: {len(uncertainty_reconstructors)} masks')
target_summary = uncertainty_results.loc[uncertainty_results['dataset'] == '2023_target']
print('Target: macro averages across masks of the same size')
display(target_summary.groupby(['n_missing', 'method'])[
    ['roc_auc', 'balanced_accuracy', 'tpr_at_1pct_fpr_roc']].mean().round(4))
print('Target ROC-AUC by mask; complete metrics are saved in comparison.csv')
display(target_summary.pivot(index='mask', columns='method', values='roc_auc').round(4))
_ = (UNCERTAINTY_DIR / 'protocol.json').write_text(json.dumps(dict(
    k=UNCERTAINTY_K, reconstruction_seed=RECONSTRUCTION_SEED, masks=MISSING_MASKS,
    n_shadow_models=N_SHADOW_MODELS, shadow_names=list(shadow_results),
    mask_selection='preselected coverage: 2 singles, 2 pairs, 2 triples, 1 quadruple; each field hidden four times',
    reconstruction='sequential random-forest leaf donor sampling; approximate predictive distribution',
    reconstruction_chain_order=MASK_FIELDS, score_std_ddof=0,
    auxiliary_year=2014, target_year=2023, target_labels_for_tuning=False,
    meta_source='half of base-attack validation record groups',
    calibration_source='other half of base-attack validation record groups',
    methods=['single', 'average', 'learned_mean', 'uncertainty'],
    queries=uncertainty_query_audits,
    limitations=['one reconstruction seed', 'no DP evaluation yet',
        'cross-year distribution shift', 'original blind-walk asymmetry retained',
        'no confirmatory significance claim; repeated shadow rows require grouped inference']),
    indent=2), encoding='utf-8')


Target: macro averages across masks of the same size


roc_auc  balanced_accuracy  tpr_at_1pct_fpr_roc
n_missing method                                                       
1         average        0.6127             0.6166               0.0028
          learned_mean   0.5279             0.5431               0.0077
          single         0.6003             0.6076               0.0032
          uncertainty    0.5216             0.5300               0.0014
2         average        0.6156             0.6183               0.0040
          learned_mean   0.4893             0.5012               0.0080
          single         0.6017             0.6084               0.0034
          uncertainty    0.4750             0.4812               0.0055
3         average        0.6056             0.5993               0.0030
          learned_mean   0.5601             0.5108               0.0041
          single         0.5831             0.5862               0.0043
          uncertainty    0.5581             0.5333               0.0074
4         average        0.6039             0.5916               0.0043
          learned_mean   0.5171             0.5018               0.0120
          single         0.5794             0.5787               0.0051
          uncertainty    0.4434             0.4977               0.0075

Target ROC-AUC by mask; complete metrics are saved in comparison.csv


method,average,learned_mean,single,uncertainty
mask,,,,
COW+SEX,0.6155,0.5206,0.6010,0.4716
MAR,0.6177,0.4737,0.6092,0.4908
MAR+COW,0.6157,0.4579,0.6024,0.4785
MAR+WKHP+COW+SEX,0.6039,0.5171,0.5794,0.4434
MAR+WKHP+SEX,0.6086,0.5586,0.5880,0.5723
WKHP,0.6077,0.5822,0.5915,0.5523
WKHP+COW+SEX,0.6026,0.5617,0.5782,0.5439
